### Stratified Sampling

In [ ]:
from sklearn.model_selection import train_test_split

# Load filtered data
df_filtered = pd.read_csv('../data/processed/filtered_complaints.csv')

# Stratified sample (10K-15K, proportional by Product)
sample_size = 12000  # Adjust
df_sample, _ = train_test_split(df_filtered, train_size=sample_size, stratify=df_filtered['Product'], random_state=42)
print(df_sample['Product'].value_counts(normalize=True))  # Check proportions

### Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Chunk each narrative, store with metadata
chunks = []
for idx, row in df_sample.iterrows():
    narrative_chunks = splitter.split_text(row['clean_narrative'])
    for chunk_idx, chunk in enumerate(narrative_chunks):
        chunks.append({
            'complaint_id': row['Complaint ID'],
            'product_category': row['Product'],  # Map to challenge categories if needed
            'product': row['Sub-product'],
            'issue': row['Issue'],
            'sub_issue': row['Sub-issue'],
            'company': row['Company'],
            'state': row['State'],
            'date_received': row['Date received'],
            'chunk_index': chunk_idx,
            'total_chunks': len(narrative_chunks),
            'text': chunk
        })

df_chunks = pd.DataFrame(chunks)
print(f"Total chunks: {len(df_chunks)}")

### Embedding and Indexing

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(df_chunks['text'].tolist(), show_progress_bar=True)
print(embeddings.shape)  # (num_chunks, 384)

# FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Save index and metadata
faiss.write_index(index, '../vector_store/faiss_index.index')
df_chunks.to_parquet('../vector_store/chunks_metadata.parquet')  # For tracing